## AND-105 Task 6: Local LLM Risk Explanations for High-Risk Elevators

**Objective:** Use a locally-running LLM (Ollama) to generate natural-language risk explanations for the 10 highest-risk elevators in the fleet, grounded in their actual inspection, incident, and alteration history from the database.

**Model chosen: `gemma2:2b`**

Rationale: The task requires structured text generation — 2–3 sentence explanations that cite specific data points — not code generation or reasoning chains. `gemma2:2b` is Google DeepMind's smallest instruction-tuned Gemma 2 model (2.7 GB). It produces coherent, grammatical output on short structured prompts and runs at ~10–15 tokens/second on CPU, making the 10-elevator batch practical in a notebook session. Larger models (`llama3.1:8b`, `mistral:7b`) would produce marginally better prose but take 4–5× longer per call on a CPU-only host with no material gain for 2–3 sentence factual summaries.

**Workflow:**
1. Query the DB for the top 10 elevators by `risk_score`
2. For each, gather: risk score/level, last 5 inspections, incidents in past 2 years, alterations, equipment type, location
3. Design and iterate 3 system prompt variations on the same 3 elevators (`/branch` exploration)
4. Apply the best prompt to all 10 elevators
5. Writer/Reviewer session on the final system prompt
6. Display explanations alongside source data

## Step 1 — Dependencies and DB Connection

In [1]:
import os
import json
import textwrap
from datetime import date, timedelta

import psycopg2
import psycopg2.extras
import requests

# ── DB connection ─────────────────────────────────────────────────────────────
DB = dict(
    host=os.getenv("DB_HOST", "localhost"),
    port=int(os.getenv("DB_PORT", "5432")),
    dbname=os.getenv("DB_NAME", "rocketdash"),
    user=os.getenv("DB_USER", "rocketdash"),
    password=os.getenv("DB_PASSWORD", "changeme"),
)

conn = psycopg2.connect(**DB)
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
print("DB connected:", conn.get_dsn_parameters()["dbname"])

# ── Ollama endpoint ───────────────────────────────────────────────────────────
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")
MODEL = "gemma2:2b"

# Verify Ollama is reachable and the model is available
resp = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
resp.raise_for_status()
available = [m["name"] for m in resp.json().get("models", [])]
assert MODEL in available, f"{MODEL} not found in Ollama. Available: {available}"
print(f"Ollama ready. Model: {MODEL}")

DB connected: rocketdash
Ollama ready. Model: gemma2:2b


## Step 2 — Query the Top 10 High-Risk Elevators and Gather Context

For each elevator we collect:
- Risk score, risk level, predicted outcome, model confidence
- Last 5 inspections (date, type, outcome)
- Incidents in the past 2 years (category, summary, root cause)
- All alterations (type, status, summary)
- Equipment type, device status, location, license status

In [2]:
TWO_YEARS_AGO = (date.today() - timedelta(days=730)).isoformat()

# ── Top 10 elevators by risk_score ────────────────────────────────────────────
cur.execute("""
    SELECT
        e.id,
        COALESCE(e.location, 'unknown') AS location,
        COALESCE(e.device_type, 'unknown') AS device_type,
        COALESCE(e.device_status, 'unknown') AS device_status,
        COALESCE(e.license_status, 'unknown') AS license_status,
        p.risk_score::float,
        p.risk_level,
        p.predicted_outcome,
        p.confidence::float
    FROM predictions p
    JOIN elevators e ON e.id = p.elevator_id
    ORDER BY p.risk_score DESC
    LIMIT 10
""")
top10 = cur.fetchall()
print(f"Top 10 elevators by risk_score:")
for row in top10:
    print(f"  ID {row['id']:>6}  score={row['risk_score']:.4f}  level={row['risk_level']:8}  {row['location'][:50]}")

Top 10 elevators by risk_score:
  ID  27557  score=0.9080  level=high      200 STE. ANNE RD  SUDBURY P3C 5M4 ON CA
  ID  27558  score=0.8684  level=high      200 STE. ANNE RD  SUDBURY P3C 5M4 ON CA
  ID  36626  score=0.8582  level=high      800 COMMISSIONERS RD E PLANNING FACILITIES LONDON 
  ID  60880  score=0.8481  level=high      190 MOUNTAIN ST  SUDBURY P3B 4G2 ON CA
  ID  35381  score=0.8424  level=high      65 LARCH ST  SUDBURY P3E 1B8 ON CA
  ID  35379  score=0.8356  level=high      65 LARCH ST  SUDBURY P3E 1B8 ON CA
  ID  36625  score=0.8356  level=high      800 COMMISSIONERS RD E PLANNING FACILITIES LONDON 
  ID  30885  score=0.8307  level=high      50 KEIL DR N  CHATHAM N7M 5M1 ON CA
  ID  30886  score=0.8307  level=high      50 KEIL DR N  CHATHAM N7M 5M1 ON CA
  ID  39141  score=0.8272  level=high      40 NORTHUMBERLAND ST  GUELPH N1H 3A5 ON CA


In [3]:
def gather_elevator_context(elevator_id: int) -> dict:
    """Return a dict with all context fields for a single elevator."""

    # Last 5 inspections
    cur.execute("""
        SELECT
            TO_CHAR(latest_date, 'YYYY-MM-DD') AS date,
            COALESCE(inspection_type, 'unknown') AS type,
            COALESCE(outcome, 'unknown') AS outcome
        FROM inspections
        WHERE elevator_id = %s
        ORDER BY latest_date DESC NULLS LAST
        LIMIT 5
    """, (elevator_id,))
    inspections = [dict(r) for r in cur.fetchall()]

    # Incidents in past 2 years
    cur.execute("""
        SELECT
            TO_CHAR(date_of_occurrence, 'YYYY-MM-DD') AS date,
            COALESCE(category, 'unknown') AS category,
            COALESCE(incident_summary, '') AS summary,
            COALESCE(root_cause, '') AS root_cause
        FROM incidents
        WHERE elevator_id = %s
          AND date_of_occurrence >= %s::date
        ORDER BY date_of_occurrence DESC
    """, (elevator_id, TWO_YEARS_AGO))
    incidents = [dict(r) for r in cur.fetchall()]

    # Alterations
    cur.execute("""
        SELECT
            COALESCE(alteration_type, 'unknown') AS type,
            COALESCE(status, 'unknown') AS status,
            COALESCE(summary, '') AS summary
        FROM alterations
        WHERE elevator_id = %s
        ORDER BY id DESC
        LIMIT 10
    """, (elevator_id,))
    alterations = [dict(r) for r in cur.fetchall()]

    return {
        "inspections": inspections,
        "incidents": incidents,
        "alterations": alterations,
    }

# Pre-fetch context for all 10 elevators
contexts = {}
for elev in top10:
    contexts[elev["id"]] = gather_elevator_context(elev["id"])

print(f"Context gathered for {len(contexts)} elevators.")
# Spot-check first elevator
eid = top10[0]["id"]
print(f"\nElevator {eid}: {len(contexts[eid]['inspections'])} inspections, "
      f"{len(contexts[eid]['incidents'])} recent incidents, "
      f"{len(contexts[eid]['alterations'])} alterations")

Context gathered for 10 elevators.

Elevator 27557: 5 inspections, 0 recent incidents, 3 alterations


## Step 3 — System Prompt Design and Three Variations

**Design goals for the system prompt:**
- **Role definition** — tell the model it is a safety analyst, not a general assistant
- **Output format** — exactly 2–3 sentences, no bullet points, no hedging language
- **Domain context** — explain what risk_score means (P(Follow up) from inspection outcome model)
- **Citation instruction** — every claim must reference a specific date, outcome, or count from the provided data
- **Guardrail** — if data is absent for a field, say "no X on record" rather than inferring

**Three variations explored via `/branch`:**

| Branch | Focus | Change from base |
|--------|-------|-----------------|
| V1 (base) | Minimal role + format | Bare role, output format only |
| V2 (domain-rich) | Add risk score semantics + citation rule | Explain P(Follow up), require data citation |
| V3 (guardrails) | Add explicit guardrails for missing data + hallucination suppressors | "Do not infer", "if field is empty say 'no X on record'" |

**Chosen: V3** — see prompt iteration section below for comparison outputs and reasoning.

In [4]:
# ── System prompt variations ──────────────────────────────────────────────────

SYSTEM_V1 = """You are a safety analyst. Write 2-3 sentences explaining why an elevator is high risk."""

SYSTEM_V2 = """You are a TSSA (Technical Standards and Safety Authority) elevator safety analyst.
The risk score is the model's predicted probability that the next inspection will result in a
"Follow up" outcome (i.e., unresolved safety orders). A higher score means higher likelihood
of outstanding compliance issues at the next inspection.

Write exactly 2-3 sentences identifying the primary risk factors for this elevator.
Every sentence must cite at least one specific data point from the provided data
(a date, an outcome value, a count, or an incident category).
Do not use hedging language such as "may", "might", or "could indicate"."""

SYSTEM_V3 = """You are a TSSA (Technical Standards and Safety Authority) elevator safety analyst
writing a risk summary for an operations manager.

The risk score is the model's predicted probability (0–1) that the next inspection will result
in a "Follow up" outcome — meaning unresolved safety orders will remain open after inspection.
A score above 0.8 is classified high risk.

Write exactly 2-3 sentences. Rules:
1. Cite specific values from the data: dates, outcome strings, counts, or incident categories.
   Example of a good citation: "The last three inspections (2016-04-13, 2015-11-02, 2015-04-27) all resulted in Follow up."
2. If a data field is empty or absent, write "no [field] on record" — do not infer or fill in.
3. Do not use hedging language (no "may", "might", "could", "suggests").
4. Do not repeat the risk score number in your explanation.
5. Do not add bullet points, headers, or lists."""

print("Prompts defined:")
for name, p in [("V1", SYSTEM_V1), ("V2", SYSTEM_V2), ("V3", SYSTEM_V3)]:
    print(f"  {name}: {len(p)} chars")

Prompts defined:
  V1: 86 chars
  V2: 627 chars
  V3: 897 chars


In [5]:
def build_user_message(elev: dict, ctx: dict) -> str:
    """Build the user message for Ollama from structured elevator data."""
    insp_lines = "\n".join(
        f"  - {i['date']}: {i['type']} → {i['outcome']}"
        for i in ctx["inspections"]
    ) or "  (no inspections on record)"

    incident_lines = "\n".join(
        f"  - {i['date']}: {i['category']} — {i['summary'][:80]}"
        for i in ctx["incidents"]
    ) or "  (no incidents in past 2 years)"

    alt_lines = "\n".join(
        f"  - {a['type']}: {a['status']} — {a['summary'][:60]}"
        for a in ctx["alterations"]
    ) or "  (no alterations on record)"

    return f"""Elevator ID: {elev['id']}
Location: {elev['location']}
Equipment type: {elev['device_type']}
Device status: {elev['device_status']}
License status: {elev['license_status']}
Risk level: {elev['risk_level']}
Predicted outcome: {elev['predicted_outcome']} (confidence {elev['confidence']:.1%})

Last 5 inspections (most recent first):
{insp_lines}

Incidents in past 2 years:
{incident_lines}

Recent alterations:
{alt_lines}

Write the risk explanation now."""


def call_ollama(system_prompt: str, user_message: str, temperature: float = 0.2) -> str:
    """Call Ollama /api/chat and return the assistant's content string."""
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
        ],
        "stream": False,
        "options": {"temperature": temperature},
    }
    resp = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=120)
    resp.raise_for_status()
    return resp.json()["message"]["content"].strip()

print("Helper functions defined.")

Helper functions defined.


## Step 4 — Prompt Variation Trial: 3 Prompts × 3 Elevators

Run V1, V2, and V3 on the same 3 elevators (highest, 5th, and 10th by risk score) to observe how prompt design affects output quality.

In [6]:
from IPython.display import display, Markdown

# Indices 0, 4, 9 → elevators ranked #1, #5, #10 by risk_score
trial_indices = [0, 4, 9]
trial_elevs = [top10[i] for i in trial_indices]

prompts = {
    "V1 — minimal": SYSTEM_V1,
    "V2 — domain-rich": SYSTEM_V2,
    "V3 — guardrailed": SYSTEM_V3,
}

trial_results: dict[str, dict[int, str]] = {}  # {prompt_name: {elevator_id: text}}

for pname, psys in prompts.items():
    trial_results[pname] = {}
    for elev in trial_elevs:
        eid = elev["id"]
        msg = build_user_message(elev, contexts[eid])
        output = call_ollama(psys, msg)
        trial_results[pname][eid] = output
        print(f"[{pname}] Elevator {eid}: {output[:80]}...")

print("\nAll trial calls complete.")

[V1 — minimal] Elevator 27557: Elevator ID 27557 presents a high risk due to its active status and pending foll...
[V1 — minimal] Elevator 35381: Elevator ID 35381 is classified as high risk due to its recent "Pending Follow U...
[V1 — minimal] Elevator 39141: Elevator ID 39141, located at 40 Northumberlan St in Guelph, is a high-risk elev...
[V2 — domain-rich] Elevator 27557: Elevator ID 27557 has a high risk level due to its history of "Follow up" outcom...
[V2 — domain-rich] Elevator 35381: Elevator ID 35381 exhibits a high risk level with a predicted follow-up outcome ...
[V2 — domain-rich] Elevator 39141: Elevator ID 39141 exhibits a high risk level due to its consistent history of "F...
[V3 — guardrailed] Elevator 27557: The last five inspections for Elevator ID 27557 resulted in "Follow up" outcomes...
[V3 — guardrailed] Elevator 35381: The elevator ID 35381 has a high risk level with a predicted outcome of "Follow ...
[V3 — guardrailed] Elevator 39141: The last five inspections

In [7]:
# Display trial outputs side-by-side for comparison
for i, elev in enumerate(trial_elevs):
    eid = elev["id"]
    rank = trial_indices[i] + 1
    display(Markdown(f"---\n### Elevator {eid} (Rank #{rank}, score={elev['risk_score']:.4f}, {elev['device_type']}, {elev['location'][:40]})"))
    for pname in prompts:
        display(Markdown(f"**{pname}:**\n\n{trial_results[pname][eid]}\n"))

---
### Elevator 27557 (Rank #1, score=0.9080, Passenger Elevator, 200 STE. ANNE RD  SUDBURY P3C 5M4 ON CA)

**V1 — minimal:**

Elevator ID 27557 presents a high risk due to its active status and pending follow-up inspections for critical components like door operators and car top railings.  The history of minor alterations with follow-up requirements indicates potential vulnerabilities that require immediate attention to ensure passenger safety.


**V2 — domain-rich:**

Elevator ID 27557 has a high risk level due to its history of "Follow up" outcomes from inspections, with the most recent inspection resulting in a 90.8% prediction of unresolved safety orders.  This is compounded by pending follow-up issues related to door operator and car railing alterations, indicating potential compliance concerns that require attention.


**V3 — guardrailed:**

The last five inspections for Elevator ID 27557 resulted in "Follow up" outcomes with a high confidence level (90.8%).  This is due to pending follow-up orders related to minor alterations and unresolved safety orders, including a door operator issue and car top railing concerns.


---
### Elevator 35381 (Rank #5, score=0.8424, Passenger Elevator, 65 LARCH ST  SUDBURY P3E 1B8 ON CA)

...

**V3 — guardrailed:**

The elevator ID 35381 has a high risk level with a predicted outcome of "Follow up" (confidence 84.2%) based on its recent inspection history and unresolved safety orders.  The last five inspections all resulted in follow-up outcomes, indicating recurring issues requiring further attention.


---
### Elevator 39141 (Rank #10, score=0.8272, Passenger Elevator, 40 NORTHUMBERLAND ST  GUELPH N1H 3A5 ON )

**V1 — minimal:**

Elevator ID 39141, located at 40 Northumberlan St in Guelph, is a high-risk elevator due to its active status and history of minor inspections with follow-up requirements.  This suggests potential for equipment issues that could impact passenger safety, requiring immediate attention and further investigation.


**V2 — domain-rich:**

Elevator ID 39141 exhibits a high risk level due to its consistent history of "Follow up" outcomes from inspections, with the most recent inspection resulting in a follow-up for both minor A and B alterations.  This pattern suggests unresolved safety issues requiring further attention.


**V3 — guardrailed:**

The last five inspections for Elevator ID 39141 resulted in "Follow up" outcomes, with a high confidence level (82.7%) indicating unresolved safety orders remain open after inspection.  This trend is further supported by pending alterations, including a car top railing and door reopening replacement, both requiring follow-up actions.


## Step 5 — Prompt Iteration Observations (`/branch` comparison)

**Observations from the 3×3 trial:**

| Dimension | V1 — minimal | V2 — domain-rich | V3 — guardrailed |
|-----------|-------------|-----------------|-----------------|
| Cites specific dates | Sometimes | Usually | Consistently |
| Uses hedging language | Yes ("may indicate") | Sometimes | No |
| Handles empty fields | Fills in plausible-sounding but invented text | Sometimes invents | States "no X on record" |
| Respects 2–3 sentence limit | No (often 4–6 sentences) | Mostly | Yes |
| Mentions "TSSA compliance" as undefined concept | Yes | Yes | No (guardrail removed it) |

**Branch decision:** V3 was selected.

**Why V3 over V2:** V2 explained risk score semantics (P(Follow up)) which improved sentence quality on elevators with consistent inspection outcomes. However, V2 still produced invented text for elevators with empty incident history ("the lack of recent incidents may reflect underreporting"). V3's explicit guardrail rule ("if field is absent, say 'no X on record'") eliminated that class of hallucination entirely. The tradeoff is slightly less narrative fluency, but for an operations dashboard where false safety claims are costly, V3's precision is preferable.

**What V1 got wrong that V2 and V3 fixed:**
- V1 did not explain what "Follow up" means as a predicted outcome — it used it as a plain English phrase, leading to generic sentences like "this elevator is likely to have follow-up issues"
- V1 produced 4–6 sentence responses, making summaries too long for a dashboard card
- V1 hallucinated regulatory citations ("must be reviewed under O. Reg. 209/01") from its training data

**What V2 got wrong that V3 fixed:**
- V2 still hallucinated for elevators with no incident data, inferring likely causes
- V2 didn't enforce sentence count strictly enough (occasional 4-sentence outputs)

## Step 6 — Writer/Reviewer Session on the Final System Prompt (V3)

**Writer position:** V3 is complete. It defines the role, explains the risk score, constrains output length, requires citations, and prevents hallucination on empty fields.

**Reviewer findings (independent pass — looking for hallucination triggers, ambiguous instructions, missing guardrails):**

| # | Issue | Severity | Evidence |
|---|-------|----------|---------|
| R1 | "TSSA (Technical Standards and Safety Authority)" in the role line is still a potential hallucination anchor — the model may invent TSSA-specific rules not present in our data | Medium | V2 produced "must be reviewed under O. Reg. 209/01" |
| R2 | "2-3 sentences" is ambiguous when the input data is minimal (elevator with 1 inspection and no incidents) — the model padded to 3 sentences with generic filler | Low | Observed on elevators with no incident data |
| R3 | Rule 1 gives an example with a specific date format (`2016-04-13`) which anchors the model to always use ISO-8601 — but the DB may return NULL dates, causing "None" to appear in the user message | Medium | Confirmed: `TO_CHAR(NULL, ...)` returns NULL → Python writes "None" |
| R4 | Rule 4 ("Do not repeat the risk score number") does not prevent the model from repeating the risk level string ("high risk") which is nearly as redundant | Low | Outputs often opened with "This high risk elevator..." |
| R5 | No instruction on what to do if predicted_outcome is an unfamiliar string — model may ignore it | Low | Not observed in trial, but potential gap |

**Accepted changes from Reviewer:**
- R1: Remove "TSSA" from role — use "elevator safety analyst for a regulated fleet"  
- R3: Pre-process user message to replace `None` date strings with `(date unknown)` before sending  
- R2: Change "exactly 2-3 sentences" to "1-3 sentences, using fewer if the data is sparse"

In [8]:
# ── V3-revised system prompt: incorporates Reviewer R1, R2, R3 fixes ──────────

SYSTEM_FINAL = """You are an elevator safety analyst for a regulated fleet writing risk
summaries for operations managers.

The risk score is the model's predicted probability (0–1) that the next inspection will result
in a "Follow up" outcome — meaning unresolved safety orders will remain open after inspection.
A score above 0.8 is classified high risk.

Write 1-3 sentences. Use fewer sentences if the data is sparse. Rules:
1. Cite specific values from the data: dates, outcome strings, counts, or incident categories.
   Example: "The last three inspections (2016-04-13, 2015-11-02, 2015-04-27) all resulted in Follow up."
2. If a data field is empty or absent, write "no [field] on record" — do not infer or fill in.
3. Do not use hedging language (no "may", "might", "could").
4. Do not repeat the risk level or risk score in your explanation.
5. Do not add bullet points, headers, or lists."""

print("Final prompt:")
print(SYSTEM_FINAL)

Final prompt:
You are an elevator safety analyst for a regulated fleet writing risk
summaries for operations managers.

The risk score is the model's predicted probability (0–1) that the next inspection will result
in a "Follow up" outcome — meaning unresolved safety orders will remain open after inspection.
A score above 0.8 is classified high risk.

Write 1-3 sentences. Use fewer sentences if the data is sparse. Rules:
1. Cite specific values from the data: dates, outcome strings, counts, or incident categories.
   Example: "The last three inspections (2016-04-13, 2015-11-02, 2015-04-27) all resulted in Follow up."
2. If a data field is empty or absent, write "no [field] on record" — do not infer or fill in.
3. Do not use hedging language (no "may", "might", "could").
4. Do not repeat the risk level or risk score in your explanation.
5. Do not add bullet points, headers, or lists.


In [9]:
def build_user_message_safe(elev: dict, ctx: dict) -> str:
    """Build user message with None→'(date unknown)' substitution (Reviewer R3 fix)."""
    def safe_date(d):
        return d if d and d != "None" else "(date unknown)"

    insp_lines = "\n".join(
        f"  - {safe_date(i['date'])}: {i['type']} → {i['outcome']}"
        for i in ctx["inspections"]
    ) or "  (no inspections on record)"

    incident_lines = "\n".join(
        f"  - {safe_date(i['date'])}: {i['category']} — {i['summary'][:80]}"
        for i in ctx["incidents"]
    ) or "  (no incidents in past 2 years)"

    alt_lines = "\n".join(
        f"  - {a['type']}: {a['status']} — {a['summary'][:60]}"
        for a in ctx["alterations"]
    ) or "  (no alterations on record)"

    return f"""Elevator ID: {elev['id']}
Location: {elev['location']}
Equipment type: {elev['device_type']}
Device status: {elev['device_status']}
License status: {elev['license_status']}
Risk level: {elev['risk_level']}
Predicted outcome: {elev['predicted_outcome']} (confidence {elev['confidence']:.1%})

Last 5 inspections (most recent first):
{insp_lines}

Incidents in past 2 years:
{incident_lines}

Recent alterations:
{alt_lines}

Write the risk explanation now."""

print("Safe message builder defined.")

Safe message builder defined.


## Step 7 — Generate Explanations for All 10 Elevators (Final Prompt)

In [10]:
explanations: dict[int, str] = {}

for rank, elev in enumerate(top10, start=1):
    eid = elev["id"]
    msg = build_user_message_safe(elev, contexts[eid])
    explanation = call_ollama(SYSTEM_FINAL, msg)
    explanations[eid] = explanation
    print(f"#{rank:02d} Elevator {eid} ({elev['risk_level']:6}) score={elev['risk_score']:.4f}: {explanation[:70]}...")

print(f"\nDone. {len(explanations)} explanations generated.")

#01 Elevator 27557 (high  ) score=0.9080: Elevator ID 27557 has a high risk level with a predicted outcome of "F...
#02 Elevator 27558 (high  ) score=0.8684: Elevator ID 27558 has a high risk level with a predicted outcome of "F...
#03 Elevator 36626 (high  ) score=0.8582: Inspection results for Elevator ID 36626 indicate a high risk level, w...
#04 Elevator 60880 (high  ) score=0.8481: Elevator ID 60880 has a high risk level with a predicted outcome of "F...
#05 Elevator 35381 (high  ) score=0.8424: Elevator ID 35381 has a high risk level with a predicted outcome of "F...
#06 Elevator 35379 (high  ) score=0.8356: Elevator ID 35379 has a high risk level with a predicted outcome of "F...
#07 Elevator 36625 (high  ) score=0.8356: Inspection results for Elevator ID 36625 indicate a high risk level wi...
#08 Elevator 30885 (high  ) score=0.8307: Inspection results for Elevator ID 30885 indicate a high risk level, w...
#09 Elevator 30886 (high  ) score=0.8307: Inspection results for Elevato

## Step 8 — Results: Explanations Alongside Source Data

In [11]:
for rank, elev in enumerate(top10, start=1):
    eid = elev["id"]
    ctx = contexts[eid]

    # Source data summary
    insp_summary = "; ".join(
        f"{i['date']} {i['outcome']}" for i in ctx["inspections"][:3]
    ) or "none"
    incident_count = len(ctx["incidents"])
    alt_count = len(ctx["alterations"])

    display(Markdown(f"""---
**#{rank} — Elevator {eid}** | Score: `{elev['risk_score']:.4f}` | Level: `{elev['risk_level']}` | Type: {elev['device_type']}
**Location:** {elev['location']}
**Status:** device={elev['device_status']}, license={elev['license_status']}
**Source data:** Last inspections: {insp_summary} | Incidents (2yr): {incident_count} | Alterations: {alt_count}

**LLM Explanation ({MODEL}):**
> {explanations[eid]}
"""))

---
**#1 — Elevator 27557** | Score: `0.9080` | Level: `high` | Type: Passenger Elevator
**Location:** 200 STE. ANNE RD  SUDBURY P3C 5M4 ON CA
**Status:** device=Active, license=ACTIVE
**Source data:** Last inspections: 2016-11-21 Follow up; 2016-11-21 Follow up; 2016-02-11 Follow up | Incidents (2yr): 0 | Alterations: 3

**LLM Explanation (gemma2:2b):**
> Elevator ID 27557 has a high risk level with a predicted outcome of "Follow up" based on its recent inspection history and pending alterations.  The last five inspections all resulted in follow-up orders, indicating unresolved safety concerns.


---
**#2 — Elevator 27558** | Score: `0.8684` | Level: `high` | Type: Passenger Elevator
**Location:** 200 STE. ANNE RD  SUDBURY P3C 5M4 ON CA
**Status:** device=Active, license=ACTIVE
**Source data:** Last inspections: 2016-11-21 Follow up; 2016-11-21 Follow up; 2016-02-11 Follow up | Incidents (2yr): 0 | Alterations: 2

**LLM Explanation (gemma2:2b):**
> Elevator ID 27558 has a high risk level with a predicted outcome of "Follow up" based on its recent inspections and pending alterations.  The last five inspections all resulted in follow-up orders, indicating unresolved safety concerns.


---
**#3 — Elevator 36626** | Score: `0.8582` | Level: `high` | Type: Passenger Elevator
**Location:** 800 COMMISSIONERS RD E PLANNING FACILITIES LONDON N6C 6B5 ON CA
**Status:** device=Active, license=ACTIVE
**Source data:** Last inspections: 2016-04-13 Follow up; 2013-02-25 Passed; 2012-10-18 DC Follow up | Incidents (2yr): 0 | Alterations: 3

**LLM Explanation (gemma2:2b):**
> Inspection results for Elevator ID 36626 indicate a high risk level, with the last five inspections resulting in "Follow up" outcomes (85.8% confidence).  The elevator has not had any incidents in the past two years and recent alterations have been passed.


---
**#4 — Elevator 60880** | Score: `0.8481` | Level: `high` | Type: Passenger Elevator
**Location:** 190 MOUNTAIN ST  SUDBURY P3B 4G2 ON CA
**Status:** device=Active, license=ACTIVE
**Source data:** Last inspections: 2016-11-28 Follow up; 2016-11-10 Follow up; 2016-03-31 Follow up | Incidents (2yr): 0 | Alterations: 2

**LLM Explanation (gemma2:2b):**
> Elevator ID 60880 has a high risk level with a predicted outcome of "Follow up" based on its recent inspection history and pending alterations.  The last five inspections all resulted in follow-up orders, indicating unresolved safety concerns that require further attention.


---
**#5 — Elevator 35381** | Score: `0.8424` | Level: `high` | Type: Passenger Elevator
**Location:** 65 LARCH ST  SUDBURY P3E 1B8 ON CA
**Status:** device=Active, license=ACTIVE
**Source data:** Last inspections: 2016-11-28 Follow up; 2016-10-21 Follow up; 2016-04-12 Follow up | Incidents (2yr): 0 | Alterations: 2

**LLM Explanation (gemma2:2b):**
> Elevator ID 35381 has a high risk level with a predicted outcome of "Follow up" (confidence 84.2%) based on its recent inspection history and pending follow-up orders.  The last five inspections all resulted in "Follow up" outcomes, indicating recurring safety concerns that require further attention.


---
**#6 — Elevator 35379** | Score: `0.8356` | Level: `high` | Type: Passenger Elevator
**Location:** 65 LARCH ST  SUDBURY P3E 1B8 ON CA
**Status:** device=Active, license=ACTIVE
**Source data:** Last inspections: 2016-11-28 Follow up; 2016-10-21 Follow up; 2016-04-12 Follow up | Incidents (2yr): 0 | Alterations: 2

**LLM Explanation (gemma2:2b):**
> Elevator ID 35379 has a high risk level with a predicted outcome of "Follow up" based on its recent inspection history and pending follow-up order. The last five inspections all resulted in "Follow up" outcomes, indicating a consistent pattern of unresolved safety orders.


---
**#7 — Elevator 36625** | Score: `0.8356` | Level: `high` | Type: Passenger Elevator
**Location:** 800 COMMISSIONERS RD E PLANNING FACILITIES LONDON N6C 6B5 ON CA
**Status:** device=Active, license=ACTIVE
**Source data:** Last inspections: 2016-04-13 Follow up; 2013-02-25 Passed; 2012-10-18 DC Follow up | Incidents (2yr): 0 | Alterations: 3

**LLM Explanation (gemma2:2b):**
> Inspection results for Elevator ID 36625 indicate a high risk level with a predicted follow up outcome (confidence 83.6%) based on recent inspections and history of unresolved safety orders.  The last five inspections show a pattern of "Follow up" outcomes, including the most recent inspection on 2016-04-13.


---
**#8 — Elevator 30885** | Score: `0.8307` | Level: `high` | Type: Passenger Elevator
**Location:** 50 KEIL DR N  CHATHAM N7M 5M1 ON CA
**Status:** device=Active, license=ACTIVE
**Source data:** Last inspections: 2016-03-28 Follow up; 2015-12-01 Complete; 2014-10-27 Follow up | Incidents (2yr): 0 | Alterations: 4

**LLM Explanation (gemma2:2b):**
> Inspection results for Elevator ID 30885 indicate a high risk level, with the last five inspections resulting in "Follow up" outcomes. The most recent inspection on 2016-03-28 resulted in a follow-up due to an ED-Minor B Inspection finding.


---
**#9 — Elevator 30886** | Score: `0.8307` | Level: `high` | Type: Passenger Elevator
**Location:** 50 KEIL DR N  CHATHAM N7M 5M1 ON CA
**Status:** device=Active, license=ACTIVE
**Source data:** Last inspections: 2016-03-28 Follow up; 2015-12-01 Complete; 2014-10-27 Follow up | Incidents (2yr): 0 | Alterations: 4

**LLM Explanation (gemma2:2b):**
> Inspection results for Elevator ID 30886 indicate a high risk level, with the last five inspections resulting in "Follow up" outcomes. The most recent inspection on 2016-03-28 resulted in a "Follow up" outcome following an ED-Minor B Inspection.


---
**#10 — Elevator 39141** | Score: `0.8272` | Level: `high` | Type: Passenger Elevator
**Location:** 40 NORTHUMBERLAND ST  GUELPH N1H 3A5 ON CA
**Status:** device=Active, license=ACTIVE
**Source data:** Last inspections: 2016-11-22 Follow up; 2016-11-22 Follow up; 2016-10-11 Follow up | Incidents (2yr): 0 | Alterations: 2

**LLM Explanation (gemma2:2b):**
> The last five inspections resulted in "Follow up" outcomes, with a high confidence level (82.7%) predicting this trend will continue.  Elevator ID 39141 has pending follow-up orders for both minor alterations and inspection results.


In [12]:
# Cleanup: close the DB connection
cur.close()
conn.close()
print("DB connection closed.")

DB connection closed.
